In [4]:
import glob
import os
import pandas as pd

In [9]:
def load_gdp_file(path):

    df = pd.read_csv(path)

    # clean column names
    df.columns = df.columns.str.strip()


    # check required column
    if "Description" not in df.columns:
        print("Skipped (no Description):", path)
        return pd.DataFrame()


    df["Description"] = (
        df["Description"]
        .astype(str)
        .str.strip()
    )


    gdp = df[
        df["Description"] == "GDP (in Rs. Cr.)"
    ].copy()


    growth = df[
        df["Description"] == "Growth Rate % (YoY)"
    ].copy()


    if gdp.empty or growth.empty:
        print("Skipped (missing GDP/Growth):", path)
        return pd.DataFrame()


    gdp.drop(
        columns=["Description"],
        inplace=True
    )

    growth.drop(
        columns=["Description"],
        inplace=True
    )


    def fix_year(y):

        y = str(y)
        start = int(y.split("-")[0])

        if start >= 1900:
            return start

        if start >= 90:
            return 1900 + start

        return 2000 + start


    gdp["Year"] = gdp["Year"].apply(fix_year)
    growth["Year"] = growth["Year"].apply(fix_year)



    gdp_long = gdp.melt(
        id_vars=["Year"],
        var_name="District",
        value_name="GDP"
    )


    growth_long = growth.melt(
        id_vars=["Year"],
        var_name="District",
        value_name="GrowthRate"
    )


    final = pd.merge(
        gdp_long,
        growth_long,
        on=["Year","District"],
        how="outer"
    )


    return final

In [10]:
def get_state_name(filename):

    name = os.path.basename(filename)

    name = name.replace("gdp_", "")
    
    # remove numbering
    name = name.replace("1","")
    name = name.replace("2","")
    name = name.replace(".csv","")

    return name.strip()

In [11]:
all_files = glob.glob(
    "data/raw/*.csv"
)

all_states = []

for file in all_files:

    print("Processing:",file)

    temp = load_gdp_file(file)

    temp["State"] = get_state_name(file)

    all_states.append(temp)

Processing: data/raw\elementary_2015_16.csv
Skipped (no Description): data/raw\elementary_2015_16.csv
Processing: data/raw\gdp_AndhraPradesh1.csv
Processing: data/raw\gdp_AndhraPradesh2.csv
Processing: data/raw\gdp_ArunachalPradesh.csv
Processing: data/raw\gdp_Assam1.csv
Processing: data/raw\gdp_Assam2.csv
Skipped (missing GDP/Growth): data/raw\gdp_Assam2.csv
Processing: data/raw\gdp_Bihar1.csv
Processing: data/raw\gdp_Bihar2.csv
Processing: data/raw\gdp_Chattisgarh.csv
Processing: data/raw\gdp_Haryana.csv
Processing: data/raw\gdp_HimachalPradesh.csv
Processing: data/raw\gdp_Jharkhand.csv
Processing: data/raw\gdp_Karnataka1.csv
Processing: data/raw\gdp_Karnataka2.csv
Processing: data/raw\gdp_Kerala1.csv
Processing: data/raw\gdp_Kerala2.csv
Processing: data/raw\gdp_MadhyaPradesh.csv
Processing: data/raw\gdp_Maharashtra1.csv
Processing: data/raw\gdp_Maharashtra2.csv
Processing: data/raw\gdp_Manipur.csv
Processing: data/raw\gdp_Meghalaya.csv
Processing: data/raw\gdp_Mizoram.csv
Processing

In [13]:
print(len(all_files))

37


In [14]:
india_df = pd.concat(
    all_states,
    ignore_index=True
)

In [15]:
print(india_df.shape)

print(india_df["State"].nunique())

print(india_df.head())

(6246, 5)
23
           State    Year       District      GDP GrowthRate
0  AndhraPradesh  1999.0       Adilabad  3463.28        NaN
1  AndhraPradesh  1999.0      Anantapur  4728.59        NaN
2  AndhraPradesh  1999.0       Chittoor  5395.57        NaN
3  AndhraPradesh  1999.0  Godavari East  9507.67        NaN
4  AndhraPradesh  1999.0  Godavari West   7398.0        NaN


In [16]:
india_df.isnull().sum()

State           0
Year            0
District        0
GDP            60
GrowthRate    910
dtype: int64

In [17]:
india_df["Year"] = (
    india_df["Year"]
    .astype(int)
)


india_df["GDP"] = (
    india_df["GDP"]
    .astype(str)
    .str.replace(",","")
    .astype(float)
)


india_df["GrowthRate"] = (
    india_df["GrowthRate"]
    .astype(float)
)

In [18]:
india_df = india_df[
    india_df["GDP"] > 0
]

In [19]:
india_df.shape

(6178, 5)

In [20]:
from sklearn.preprocessing import MinMaxScaler

In [21]:
scaler = MinMaxScaler()


india_df["GDP_score"] = (
    scaler.fit_transform(
        india_df[["GDP"]]
    )
)


india_df["Growth_score"] = (
    scaler.fit_transform(
        india_df[["GrowthRate"]]
        .fillna(0)
    )
)

In [22]:
india_df["DDI"] = (
    0.6 * india_df["GDP_score"]
    +
    0.4 * india_df["Growth_score"]
)

In [23]:
india_df.to_csv(
"data/cleaned/India_District_Development.csv",
index=False
)

In [24]:
india_df.sort_values("DDI",ascending=False).head()

,State,Year,District,GDP,GrowthRate,GDP_score,Growth_score,DDI
3186,Maharashtra,2012,Mumbai,191910.0,7.630788,1.000000,0.422616,0.769046
3050,Maharashtra,2008,Mumbai,153339.0,63.364691,0.798988,0.650723,0.739682
3152,Maharashtra,2011,Mumbai,178304.0,6.187691,0.929092,0.416710,0.724139
3118,Maharashtra,2010,Mumbai,167914.0,9.455830,0.874945,0.430085,0.697001
3084,Maharashtra,2009,Mumbai,153408.0,0.044998,0.799347,0.391569,0.636236


In [26]:
india_df["Rank"] = (
    india_df
    .groupby("Year")["DDI"]
    .rank(
        ascending=False,
        method="dense"
    )
)

In [27]:
india_df.sort_values(
    ["Year","Rank"]
).head(20)

,State,Year,District,GDP,GrowthRate,GDP_score,Growth_score,DDI,Rank
2676,Maharashtra,1999,Mumbai,54374.09,NaN,0.283233,0.391385,0.326494,1.0
2690,Maharashtra,1999,Thane,28755.60,NaN,0.149723,0.391385,0.246387,2.0
2683,Maharashtra,1999,Pune,27231.84,NaN,0.141782,0.391385,0.241623,3.0
1769,Karnataka,1999,Bangalore Urban,21876.93,NaN,0.113875,0.391385,0.224879,4.0
5961,WestBengal,1999,24-Parganas (North),15562.52,NaN,0.080967,0.391385,0.205134,5.0
5965,WestBengal,1999,Burdwan,13752.99,NaN,0.071537,0.391385,0.199476,6.0
4551,Tamilnadu,1999,Chennai,13215.12,NaN,0.068734,0.391385,0.197794,7.0
5972,WestBengal,1999,Kolkata,12694.24,NaN,0.066019,0.391385,0.196165,8.0
2680,Maharashtra,1999,Nashik,12422.13,NaN,0.064601,0.391385,0.195314,9.0
4552,Tamilnadu,1999,Coimbatore,11543.99,NaN,0.060025,0.391385,0.192569,10.0


In [28]:
latest_year = india_df["Year"].max()

latest_df = india_df[
    india_df["Year"] == latest_year
]

latest_df.shape

(71, 9)

In [29]:
def category(score):

    if score >= 0.75:
        return "High Development"

    elif score >= 0.45:
        return "Medium Development"

    else:
        return "Low Development"


india_df["Category"] = india_df["DDI"].apply(category)

In [30]:
state_summary = (
    india_df
    .groupby("State")
    .agg(
        Avg_DDI=("DDI","mean"),
        Avg_GDP=("GDP","mean"),
        Avg_Growth=("GrowthRate","mean"),
        Districts=("District","nunique")
    )
    .reset_index()
)


state_summary.sort_values(
    "Avg_DDI",
    ascending=False
).head(10)

,State,Avg_DDI,Avg_GDP,Avg_Growth,Districts
11,Maharashtra,0.211515,13128.254577,9.767934,34
0,AndhraPradesh,0.202923,10882.336117,8.542370,23
22,WestBengal,0.199888,11020.090143,6.334520,19
9,Kerala,0.196195,9273.949732,7.486753,14
8,Karnataka,0.181215,5828.317296,4.889369,30
16,Punjab,0.180982,5467.783889,5.307212,20
19,Tamilnadu,0.180177,5210.541792,5.174795,30
5,Haryana,0.177517,3430.062406,7.354157,19
17,Rajasthan,0.177364,3797.188301,6.523096,33
20,UttarPradesh,0.176582,3806.045521,5.793301,74


In [31]:
india_df.to_csv(
    "data/processed/district_development_index.csv",
    index=False
)


state_summary.to_csv(
    "data/processed/state_summary.csv",
    index=False
)